# Q2. Distance Matrix Computation

This notebook demonstrates how to load, clean, and preprocess the dataset (`lab2 data.xlsx`) before computing a pairwise distance matrix using two different metrics: **Euclidean** and **Manhattan**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

### 1. Load and Inspect Dataset

In [2]:
df = pd.read_excel('lab2 data.xlsx')
print("Original Shape:", df.shape)
df.head()

Original Shape: (163, 14)


,Timestamp,Reg No,Job role that you are interested in,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Which is the highest paying company,Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester,Internships Interests
0,2024-12-04 20:31:09,2341338,Data Scientist,3 LPA,1Cr,20 LPA,Deolite,3.0,4,20 LPA,0.6,0.62,0.77,Industry
1,2024-12-04 20:33:07,2341310,Data Scientist,5,NaN,NaN,DE Shaw,3.0,2,11,67,60,97,Industry
2,2024-12-04 20:39:43,2341311,Data Scientist,7 LPA,15 LPA,8 LPA,DE Shaw,3.0,4,5 LPA,80,80,99,Research
3,2024-12-04 20:40:37,2341324,Data Scientist,3 LPA,25 LPA,10 LPA,DE Shaw,NaN,3,20,75,70,95,Research
4,2024-12-04 20:41:51,2341324,Business Analyst,2,50,25,Deolite,3.0,3,20,65,76,96,Industry


### 2. Data Cleaning
We drop irrelevant columns like `Timestamp` and `Reg No`. We also ensure that all salary, GPA, and percentage columns are extracted as purely numerical values, as respondents might have included characters like 'LPA' or '%'.

In [3]:
# Drop irrelevant identification and timestamp columns
df_cleaned = df.drop(columns=['Timestamp', 'Reg No'], errors='ignore')

# Identify columns that should be purely numeric
numeric_cols = [
    'What is the minimum salary of students placed through campus (In LPA..respond as a number)',
    'What is the maximum salary of students placed through campus (In LPA..respond as a number)',
    'What is the  median salary of students placed through campus (In LPA..respond as a number)',
    'Rate your contribution towards extra curricular activities',
    'Rate your technical competencies',
    'What are your package expectations (LPA)',
    'your CIA % of last semester',
    'your GPA of last semester',
    'Your maximum attendance % till last semester'
]

# Extract numbers and convert to float (handles cases like '10 LPA' or '85%')
for col in numeric_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

# Fill missing values for numerical columns with their median
for col in numeric_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# For categorical columns, fill missing with the most frequent value (mode)
categorical_cols = df_cleaned.select_dtypes(exclude=['number', 'datetime']).columns
for col in categorical_cols:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

# One-hot encode the categorical variables
df_encoded = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)

print("Cleaned and Encoded Data Shape:", df_encoded.shape)
df_encoded.head()

Cleaned and Encoded Data Shape: (163, 16)


,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester,Job role that you are interested in_Data Engineer,Job role that you are interested in_Data Scientist,Job role that you are interested in_Software Engineer,Which is the highest paying company_DE Shaw,Which is the highest paying company_Deolite,Which is the highest paying company_EY,Internships Interests_Research
0,3.0,1.0,20.0,3.0,4.0,20.0,0.6,0.62,0.77,False,True,False,False,True,False,False
1,5.0,22.0,9.5,3.0,2.0,11.0,67.0,60.00,97.00,False,True,False,True,False,False,False
2,7.0,15.0,8.0,3.0,4.0,5.0,80.0,80.00,99.00,False,True,False,True,False,False,True
3,3.0,25.0,10.0,3.0,3.0,20.0,75.0,70.00,95.00,False,True,False,True,False,False,True
4,2.0,50.0,25.0,3.0,3.0,20.0,65.0,76.00,96.00,False,False,False,False,True,False,False


### 3. Feature Scaling
Because features like Salary and GPA are on vastly different scales, distance metrics will be artificially dominated by features with higher magnitudes. Therefore, we use **StandardScaler** to normalize the dataset before calculation.

In [4]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)

# To keep the output readable, we'll compute the distance matrix on a subset of students (e.g., the first 5 students)
X_subset = df_scaled.head(5)
print("Sample of scaled data used for distance matrix:")
X_subset

Sample of scaled data used for distance matrix:


,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),your CIA % of last semester,your GPA of last semester,Your maximum attendance % till last semester,Job role that you are interested in_Data Engineer,Job role that you are interested in_Data Scientist,Job role that you are interested in_Software Engineer,Which is the highest paying company_DE Shaw,Which is the highest paying company_Deolite,Which is the highest paying company_EY,Internships Interests_Research
0,-0.202753,-0.126846,-0.217374,-0.344753,0.778303,-0.212885,-2.195152,-0.378706,-2.304955,-0.474936,1.276335,-0.551362,-0.900617,1.783112,-0.415526,-0.560817
1,-0.202746,-0.126843,-0.217391,-0.344753,-1.487115,-0.212913,0.049467,-0.003545,0.574392,-0.474936,1.276335,-0.551362,1.110350,-0.560817,-0.415526,-0.560817
2,-0.202740,-0.126844,-0.217393,-0.344753,0.778303,-0.212931,0.488925,0.122815,0.634235,-0.474936,1.276335,-0.551362,1.110350,-0.560817,-0.415526,1.783112
3,-0.202753,-0.126843,-0.217390,-0.344753,-0.354406,-0.212885,0.319902,0.059635,0.514549,-0.474936,1.276335,-0.551362,1.110350,-0.560817,-0.415526,1.783112
4,-0.202756,-0.126840,-0.217366,-0.344753,-0.354406,-0.212885,-0.018142,0.097543,0.544470,-0.474936,-0.783493,-0.551362,-0.900617,1.783112,-0.415526,-0.560817


### 4. Distance Matrix Computation
We calculate the pairwise distance matrix using two metrics: **Euclidean** and **Manhattan**.

In [5]:
# 1. Euclidean Distance
euclidean_dist = pairwise_distances(X_subset, metric='euclidean')

print("\n--- Euclidean Distance Matrix (First 5 students) ---")
print(pd.DataFrame(euclidean_dist, index=[f"Student {i+1}" for i in range(5)], columns=[f"Student {i+1}" for i in range(5)]))


--- Euclidean Distance Matrix (First 5 students) ---
           Student 1  Student 2  Student 3  Student 4  Student 5
Student 1   0.000000   5.304697   5.579124   5.548177   4.314085
Student 2   5.304697   0.000000   3.292233   2.618729   3.883246
Student 3   5.579124   3.292233   0.000000   1.153220   4.563303
Student 4   5.548177   2.618729   1.153220   0.000000   4.403577
Student 5   4.314085   3.883246   4.563303   4.403577   0.000000


In [6]:
# 2. Manhattan Distance
manhattan_dist = pairwise_distances(X_subset, metric='manhattan')

print("\n--- Manhattan Distance Matrix (First 5 students) ---")
print(pd.DataFrame(manhattan_dist, index=[f"Student {i+1}" for i in range(5)], columns=[f"Student {i+1}" for i in range(5)]))


--- Manhattan Distance Matrix (First 5 students) ---
           Student 1  Student 2  Student 3  Student 4  Student 5
Student 1   0.000000  12.119495  12.823692  13.604452   8.695239
Student 2  12.119495   0.000000   5.235037   3.870133   7.746118
Student 3  12.823692   5.235037   0.000000   1.484661  10.513561
Student 4  13.604452   3.870133   1.484661   0.000000   9.164558
Student 5   8.695239   7.746118  10.513561   9.164558   0.000000
